# Weaver: disorganized vs. organized complexity with desire paths

We keep the **same walkers, same landscape, same destinations, and same movement problem**.

Only one mechanism changes:

- **Scenario A — no memory:** walkers do not use traces left by previous walkers.
- **Scenario C — memory + feedback:** repeated use makes some cells into routes, and later walkers preferentially use those routes.

The key contrast is:

> **Disorganized complexity:** individual trajectories vary, but aggregate traffic spread is stable.

> **Organized complexity:** past actions create persistent structure, so the particular history of interaction matters.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

WIDTH = 41
HEIGHT = 31

buildings = [
    (3, 5),
    (3, 25),
    (37, 15)
]


## Helper: moves that get a walker closer to its destination


In [ ]:
def closer_neighbors(x, y, gx, gy):
    current_distance = (x - gx)**2 + (y - gy)**2
    candidates = []

    for dx, dy in [
        (1, 0), (-1, 0), (0, 1), (0, -1),
        (1, 1), (1, -1), (-1, 1), (-1, -1)
    ]:
        nx = x + dx
        ny = y + dy

        if 0 <= nx < WIDTH and 0 <= ny < HEIGHT:
            new_distance = (nx - gx)**2 + (ny - gy)**2
            if new_distance < current_distance:
                candidates.append((nx, ny))

    return candidates


## One simulation function

The same system can be run in two modes.

- `memory=False`: walkers choose independently among reasonable steps.
- `memory=True`: walkers usually prefer cells that have accumulated more use.

A cell becomes a **route** once its popularity crosses a threshold.


In [ ]:
def run_desire_path_simulation(
    memory=False,
    seed=123,
    n_walkers=60,
    n_ticks=400,
    follow_probability=0.85,
    route_threshold=18
):
    rng = np.random.default_rng(seed)

    footfall = np.zeros((HEIGHT, WIDTH), dtype=int)
    popularity = np.zeros((HEIGHT, WIDTH), dtype=float)

    positions = []
    goals = []

    for _ in range(n_walkers):
        start, goal = rng.choice(len(buildings), size=2, replace=False)
        positions.append(buildings[start])
        goals.append(buildings[goal])

    route_history = []

    for _ in range(n_ticks):
        for i in range(n_walkers):

            x, y = positions[i]
            gx, gy = goals[i]

            if (x, y) == (gx, gy):
                current = buildings.index((gx, gy))
                possible = [j for j in range(len(buildings)) if j != current]
                goals[i] = buildings[rng.choice(possible)]
                continue

            candidates = closer_neighbors(x, y, gx, gy)

            if not candidates:
                continue

            if not memory:
                nx, ny = candidates[rng.integers(len(candidates))]

            else:
                if rng.random() < follow_probability:
                    values = [popularity[cy, cx] for cx, cy in candidates]
                    best_value = max(values)
                    best_cells = [
                        cell for cell, value in zip(candidates, values)
                        if value == best_value
                    ]
                    nx, ny = best_cells[rng.integers(len(best_cells))]
                else:
                    nx, ny = candidates[rng.integers(len(candidates))]

            positions[i] = (nx, ny)
            footfall[ny, nx] += 1

            if memory:
                popularity[ny, nx] += 1

        is_route = popularity >= route_threshold
        route_history.append(is_route.sum())

    is_route = popularity >= route_threshold
    return np.array(route_history), footfall, is_route


# Scenario A — Disorganized complexity

We summarize each run by the **entropy of foot traffic**.

Higher entropy means traffic is spread more evenly across the landscape.
Lower entropy means traffic is concentrated in fewer places.


In [ ]:
def traffic_spread(footfall):
    p = footfall.flatten()
    p = p[p > 0]
    p = p / p.sum()
    return -(p * np.log(p)).sum()


In [ ]:
seeds_to_try = [1, 2, 3, 4]

scenario_A_spread = []

for seed in seeds_to_try:
    _, footfall_A, _ = run_desire_path_simulation(
        memory=False,
        n_walkers=60,
        n_ticks=400,
        seed=seed
    )

    scenario_A_spread.append(
        traffic_spread(footfall_A)
    )

print("Scenario A — traffic spread across four runs:")
print([f"{s:.4f}" for s in scenario_A_spread])


### Interpretation

The microscopic trajectories differ from run to run.

But if the entropy values remain very close, the **aggregate spread of traffic is statistically stable**.

That is the intuition we want from Weaver's **disorganized complexity**.


# Scenario C — Organized complexity

Now past use changes future movement.

We record both:

1. **how many permanent route cells form**, and
2. **where those route cells form**.


In [ ]:
scenario_C_route_counts = []
scenario_C_route_locations = []
scenario_C_footfalls = []

for seed in seeds_to_try:
    route_history_C, footfall_C, is_route_C = run_desire_path_simulation(
        memory=True,
        n_walkers=60,
        n_ticks=400,
        seed=seed
    )

    scenario_C_route_counts.append(int(route_history_C[-1]))
    scenario_C_route_locations.append(set(zip(*np.where(is_route_C))))
    scenario_C_footfalls.append(footfall_C)

print("Scenario C — permanent paths formed across four runs:")
print(scenario_C_route_counts)


## Do the same routes form every time?

We compare route locations using **Jaccard similarity**.

- `1` = exactly the same route cells
- `0` = no route cells in common


In [ ]:
def jaccard(a, b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

for (i, a), (j, b) in combinations(
    enumerate(scenario_C_route_locations, start=1),
    2
):
    print(
        f"Run {i} vs Run {j}: {jaccard(a, b):.3f}"
    )


## Visualize the four organized outcomes


In [ ]:
for seed, footfall, routes in zip(
    seeds_to_try,
    scenario_C_footfalls,
    scenario_C_route_locations
):
    plt.figure(figsize=(8, 6))
    plt.imshow(footfall, origin="lower")

    for x, y in buildings:
        plt.scatter(x, y, marker="s", s=90)

    if routes:
        route_y, route_x = zip(*routes)
        plt.scatter(
            route_x,
            route_y,
            s=12,
            facecolors="none",
            edgecolors="black"
        )

    plt.title(f"Organized complexity — seed {seed}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()


# What should we learn?

## Scenario A — Disorganized complexity

Different individual trajectories occur, but a simple aggregate property — **traffic spread** — remains very similar across runs.

## Scenario C — Organized complexity

Past movement changes future movement:

**movement → popularity → route formation → future movement**

The number and location of permanent routes can differ substantially across runs.

So the key difference is not more walkers or more randomness.

> **History becomes structure, and structure feeds back into behavior.**

That is the central intuition we want students to carry from Weaver.
